In [ ]:
!pip install torch numpy matplotlib -q

import re
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from tkinter import Tk, filedialog
from torch.utils.data import DataLoader, TensorDataset


# ============================================================
# 1. LOAD DATASET
# ============================================================

print("Select the English file...")

root = Tk()
root.withdraw()

english_file = filedialog.askopenfilename(
    title="Select En-Ta English.txt",
    filetypes=[("Text files", "*.txt")]
)

print("Select the Tamil file...")

tamil_file = filedialog.askopenfilename(
    title="Select En-Ta Tamil.txt",
    filetypes=[("Text files", "*.txt")]
)

root.destroy()

with open(english_file, "r", encoding="utf-8") as f:
    english_lines = f.readlines()

with open(tamil_file, "r", encoding="utf-8") as f:
    tamil_lines = f.readlines()

data = []

for english, tamil in zip(english_lines, tamil_lines):

    english = english.strip()
    tamil = tamil.strip()

    if english and tamil:
        data.append((english, tamil))

data = data[:10000]

print("Total sentence pairs:", len(data))


# ============================================================
# 2. TOKENIZATION
# ============================================================

def tokenize_english(text):
    return re.findall(r"\w+|[?.!,]", text.lower())


def tokenize_tamil(text):
    return text.strip().split()


eng_tokens = [
    tokenize_english(x[0])
    for x in data
]

tam_tokens = [
    tokenize_tamil(x[1])
    for x in data
]


# ============================================================
# 3. CREATE VOCABULARIES
# ============================================================

special_tokens = [
    "<PAD>",
    "<START>",
    "<END>",
    "<UNK>"
]

eng_words = special_tokens + sorted(
    set(
        word
        for sentence in eng_tokens
        for word in sentence
    )
)

tam_words = special_tokens + sorted(
    set(
        word
        for sentence in tam_tokens
        for word in sentence
    )
)

eng_vocab = {
    word: i
    for i, word in enumerate(eng_words)
}

tam_vocab = {
    word: i
    for i, word in enumerate(tam_words)
}

PAD_IDX = tam_vocab["<PAD>"]


# ============================================================
# 4. NUMERICALIZE SENTENCES
# ============================================================

def numericalize(tokens, vocab):

    return (
        [vocab["<START>"]]
        +
        [
            vocab.get(word, vocab["<UNK>"])
            for word in tokens
        ]
        +
        [vocab["<END>"]]
    )


eng_seq = [
    numericalize(x, eng_vocab)
    for x in eng_tokens
]

tam_seq = [
    numericalize(x, tam_vocab)
    for x in tam_tokens
]


# ============================================================
# 5. PADDING
# ============================================================

max_eng = max(
    len(x)
    for x in eng_seq
)

max_tam = max(
    len(x)
    for x in tam_seq
)


def pad_sequences(
    sequences,
    max_len,
    pad_idx
):

    return [
        seq + [pad_idx] * (max_len - len(seq))
        for seq in sequences
    ]


eng_data = torch.tensor(
    pad_sequences(
        eng_seq,
        max_eng,
        eng_vocab["<PAD>"]
    )
)

tam_data = torch.tensor(
    pad_sequences(
        tam_seq,
        max_tam,
        tam_vocab["<PAD>"]
    )
)


# ============================================================
# 6. DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# 7. ENCODER
# ============================================================

class Encoder(nn.Module):

    def __init__(
        self,
        input_dim,
        emb_dim,
        hidden_dim
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            input_dim,
            emb_dim
        )

        self.rnn = nn.GRU(
            emb_dim,
            hidden_dim,
            batch_first=True
        )


    def forward(self, x):

        embedded = self.embedding(x)

        outputs, hidden = self.rnn(
            embedded
        )

        return outputs, hidden


# ============================================================
# 8. ATTENTION
# ============================================================

class Attention(nn.Module):

    def __init__(
        self,
        hidden_dim
    ):

        super().__init__()

        self.attn = nn.Linear(
            hidden_dim * 2,
            hidden_dim
        )

        self.v = nn.Linear(
            hidden_dim,
            1,
            bias=False
        )


    def forward(
        self,
        hidden,
        encoder_outputs
    ):

        hidden = hidden[-1].unsqueeze(1)

        hidden = hidden.repeat(
            1,
            encoder_outputs.size(1),
            1
        )

        energy = torch.tanh(
            self.attn(
                torch.cat(
                    (
                        hidden,
                        encoder_outputs
                    ),
                    dim=2
                )
            )
        )

        scores = self.v(
            energy
        ).squeeze(2)

        return torch.softmax(
            scores,
            dim=1
        )


# ============================================================
# 9. DECODER
# ============================================================

class Decoder(nn.Module):

    def __init__(
        self,
        output_dim,
        emb_dim,
        hidden_dim,
        attention
    ):

        super().__init__()

        self.output_dim = output_dim

        self.embedding = nn.Embedding(
            output_dim,
            emb_dim
        )

        self.attention = attention

        self.rnn = nn.GRU(
            emb_dim + hidden_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            emb_dim + hidden_dim * 2,
            output_dim
        )


    def forward(
        self,
        input_token,
        hidden,
        encoder_outputs
    ):

        embedded = self.embedding(
            input_token
        ).unsqueeze(1)

        attn = self.attention(
            hidden,
            encoder_outputs
        )

        context = torch.bmm(
            attn.unsqueeze(1),
            encoder_outputs
        )

        output, hidden = self.rnn(
            torch.cat(
                (
                    embedded,
                    context
                ),
                dim=2
            ),
            hidden
        )

        prediction = self.fc(
            torch.cat(
                (
                    output,
                    context,
                    embedded
                ),
                dim=2
            )
        )

        return (
            prediction.squeeze(1),
            hidden,
            attn
        )


# ============================================================
# 10. SEQ2SEQ MODEL
# ============================================================

class Seq2Seq(nn.Module):

    def __init__(
        self,
        encoder,
        decoder
    ):

        super().__init__()

        self.encoder = encoder
        self.decoder = decoder


    def forward(
        self,
        src,
        trg,
        teacher_forcing_ratio=0.5
    ):

        batch_size = src.size(0)

        trg_len = trg.size(1)

        output_dim = self.decoder.output_dim

        outputs = torch.zeros(
            batch_size,
            trg_len,
            output_dim
        ).to(device)

        encoder_outputs, hidden = self.encoder(
            src
        )

        input_token = trg[:, 0]

        for t in range(
            1,
            trg_len
        ):

            output, hidden, _ = self.decoder(
                input_token,
                hidden,
                encoder_outputs
            )

            outputs[:, t] = output

            best_guess = output.argmax(1)

            input_token = (
                trg[:, t]
                if torch.rand(1).item()
                < teacher_forcing_ratio
                else best_guess
            )

        return outputs


# ============================================================
# 11. CREATE MODEL
# ============================================================

encoder = Encoder(
    len(eng_vocab),
    128,
    256
)

attention = Attention(
    256
)

decoder = Decoder(
    len(tam_vocab),
    128,
    256,
    attention
)

model = Seq2Seq(
    encoder,
    decoder
).to(device)


# ============================================================
# 12. OPTIMIZER AND LOSS
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)


# ============================================================
# 13. CREATE BATCHES
# ============================================================

BATCH_SIZE = 32

dataset = TensorDataset(
    eng_data,
    tam_data
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)


# ============================================================
# 14. TRAIN MODEL
# ============================================================

EPOCHS = 100

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for batch_eng, batch_tam in loader:

        batch_eng = batch_eng.to(device)
        batch_tam = batch_tam.to(device)

        optimizer.zero_grad()

        output = model(
            batch_eng,
            batch_tam
        )

        output_dim = output.shape[-1]

        output = output[
            :, 1:
        ].reshape(
            -1,
            output_dim
        )

        target = batch_tam[
            :, 1:
        ].reshape(
            -1
        )

        loss = criterion(
            output,
            target
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(loader)

    if (epoch + 1) % 20 == 0:

        print(
            f"Epoch {epoch + 1}/{EPOCHS} "
            f"Loss: {average_loss:.4f}"
        )


# ============================================================
# 15. REVERSE TAMIL VOCABULARY
# ============================================================

tam_idx_to_word = {
    i: word
    for word, i in tam_vocab.items()
}


# ============================================================
# 16. TRANSLATION FUNCTION
# ============================================================

def translate(sentence):

    model.eval()

    tokens = tokenize_english(
        sentence
    )

    sequence = numericalize(
        tokens,
        eng_vocab
    )

    src = torch.tensor(
        sequence
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        encoder_outputs, hidden = model.encoder(
            src
        )

    input_token = torch.tensor(
        [
            tam_vocab["<START>"]
        ]
    ).to(device)

    result = []

    attention_values = []

    for _ in range(max_tam):

        with torch.no_grad():

            output, hidden, attn = model.decoder(
                input_token,
                hidden,
                encoder_outputs
            )

        prediction = output.argmax(
            1
        ).item()

        attention_values.append(
            attn.squeeze(0)
            .cpu()
            .numpy()
        )

        if prediction == tam_vocab["<END>"]:
            break

        if prediction not in [
            tam_vocab["<PAD>"],
            tam_vocab["<START>"]
        ]:

            result.append(
                tam_idx_to_word[prediction]
            )

        input_token = torch.tensor(
            [prediction]
        ).to(device)

    return (
        " ".join(result),
        np.array(attention_values)
    )


# ============================================================
# 17. TEST MULTIPLE SENTENCES
# ============================================================

test_sentences = [
    "How are you?",
    "What is your name?",
    "Where are you going?",
    "I am fine.",
    "Thank you.",
    "Good morning.",
    "Good night.",
    "I like you."
]

print("\n" + "=" * 50)
print("TRANSLATION RESULTS")
print("=" * 50)

for test_sentence in test_sentences:

    translation, _ = translate(
        test_sentence
    )

    print(
        "English:",
        test_sentence
    )

    print(
        "Tamil:",
        translation
    )

    print()


# ============================================================
# 18. SINGLE SENTENCE FOR ATTENTION VISUALIZATION
# ============================================================

test_sentence = "How are you?"

translation, attention_weights = translate(
    test_sentence
)

print("=" * 50)
print("ATTENTION VISUALIZATION")
print("=" * 50)

print(
    "English:",
    test_sentence
)

print(
    "Tamil:",
    translation
)


# ============================================================
# 19. ATTENTION VISUALIZATION
# ============================================================

english_words = (
    ["<START>"]
    + tokenize_english(test_sentence)
    + ["<END>"]
)

tamil_words = translation.split()

attention_to_plot = attention_weights[
    :len(tamil_words)
]

attention_to_plot = attention_to_plot[
    :, :len(english_words)
]

plt.figure(
    figsize=(10, 6)
)

plt.imshow(
    attention_to_plot,
    aspect="auto"
)

plt.xticks(
    range(len(english_words)),
    english_words,
    rotation=45
)

plt.yticks(
    range(len(tamil_words)),
    tamil_words
)

plt.xlabel(
    "English Words"
)

plt.ylabel(
    "Tamil Words"
)

plt.title(
    "Attention Visualization"
)

plt.tight_layout()

plt.show()

Select the English file...
Select the Tamil file...
Total sentence pairs: 8902
Device: cpu
